<a href="https://colab.research.google.com/github/Andrei-WongE/advanced_geospatial_methods/blob/origin/GeoNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

4. Example on real data from Authors, in google colab

---



In [ ]:
# FIXED: Complete geospaNN Example_realdata
!pip install geospaNN geopandas[all] scipy seaborn matplotlib shapely -q
!apt update -qq && apt install -y wget unzip -qq  # For data

import torch
import geospaNN
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy import spatial, interpolate
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Download data
os.makedirs('./data', exist_ok=True)
!wget -q -O data/covariate0605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/covariate0605.csv
!wget -q -O data/pm25_0605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/pm25_0605.csv
!wget -q -O data/Normalized_PM2.5_20190605.csv https://raw.githubusercontent.com/WentaoZhan1998/geospaNN/main/data/Normalized_PM2.5_20190605.csv

print("Starting geospaNN Example_realdata...")

# Load US boundaries
url = "https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_nation_20m.zip"
us = gpd.read_file(url).explode()
us = us.loc[us.geometry.apply(lambda x: x.exterior.bounds[2]) < -60]

# Load data
df_covariates = pd.read_csv('./data/covariate0605.csv')
df_pm25 = pd.read_csv('./data/pm25_0605.csv')
df_pm25 = df_pm25.loc[df_pm25.Latitude < 50]

# Grid setup
x_min, y_min, x_max, y_max = np.array([np.min(df_covariates['long']), np.min(df_covariates['lat']),
                                      np.max(df_covariates['long']), np.max(df_covariates['lat'])])
arr1 = np.mgrid[x_min:x_max:101j, y_min:y_max:101j]
arr1x = np.ravel(arr1[0])
arr1y = np.ravel(arr1[1])
df = pd.DataFrame({'X': arr1x, 'Y': arr1y})
df['coords'] = list(zip(df['X'], df['Y']))
df['coords'] = df['coords'].apply(Point)
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(x=df.X, y=df.Y), crs=us.crs)
inUS = gdf['geometry'].apply(lambda s: s.within(us.geometry.unary_union))

# Interpolate PM2.5
lonlat_pm25 = df_pm25.values[:, [1, 2]]
near = df_covariates.values[:, [1, 2]]
tree = spatial.KDTree(list(zip(near[:, 0].ravel(), near[:, 1].ravel())))
idx = tree.query(lonlat_pm25)[1]
df_pm25_mean = df_pm25.assign(neighbor=idx).groupby('neighbor')['PM25'].mean()
idx_new = df_pm25_mean.index.values
pm25 = df_pm25_mean.values
z = pm25[:, None]
lon = df_covariates.values[:, 1]
lat = df_covariates.values[:, 2]
f = interpolate.Rbf(lon[idx_new], lat[idx_new], z, function='inverse')
x_test = gdf.loc[inUS, :].X
y_test = gdf.loc[inUS, :].Y
z_test = f(x_test, y_test)

# Plot 1: Interpolated PM2.5
plt.figure(figsize=(9, 5))
c = plt.scatter(x=x_test, y=y_test, s=10, c=z_test, marker='s', alpha=0.7)
plt.plot(np.array(df_pm25['Longitude']), np.array(df_pm25['Latitude']), 'o', c='orange', markersize=4)
plt.colorbar(c)
plt.title('Interpolated PM2.5')
plt.show()

# Load normalized data
data_PM25 = pd.read_csv("./data/Normalized_PM2.5_20190605.csv")
X = torch.from_numpy(data_PM25[['precipitation', 'temperature', 'air pressure', 'relative humidity', 'U-wind', 'V-wind']].to_numpy()).float()
Y = torch.from_numpy(data_PM25[['PM 2.5']].to_numpy().reshape(-1)).float()
coord = torch.from_numpy(data_PM25[['longitude', 'latitude']].to_numpy()).float()

p = X.shape[1]
n = X.shape[0]
nn = 20

X, Y, coord, _ = geospaNN.spatial_order(X, Y, coord, method='max-min')
data = geospaNN.make_graph(X, Y, coord, nn)

torch.manual_seed(2024)
np.random.seed(0)
data_train, data_val, data_test = geospaNN.split_data(X, Y, coord, neighbor_size=20, test_proportion=0.5)

# Train NN
print("Training NN...")
mlp_nn = torch.nn.Sequential(
    torch.nn.Linear(p, 50), torch.nn.ReLU(),
    torch.nn.Linear(50, 20), torch.nn.ReLU(),
    torch.nn.Linear(20, 1)
)
nn_model = geospaNN.nn_train(mlp_nn, lr=0.01, min_delta=0.001)
nn_model.train(data_train, data_val, data_test)

# Train NNGLS
print("Training NNGLS...")
start_time = time.time()
theta0 = geospaNN.theta_update(torch.tensor([1, 1.5, 0.01]), mlp_nn(data_train.x).squeeze() - data_train.y, data_train.pos, neighbor_size=20)
mlp_nngls = torch.nn.Sequential(
    torch.nn.Linear(p, 100), torch.nn.ReLU(),
    torch.nn.Linear(100, 50), torch.nn.ReLU(),
    torch.nn.Linear(50, 20), torch.nn.ReLU(),
    torch.nn.Linear(20, 10), torch.nn.ReLU(),
    torch.nn.Linear(10, 1)
)
model = geospaNN.nngls(p=p, neighbor_size=nn, coord_dimensions=2, mlp=mlp_nngls, theta=torch.tensor(theta0))
nngls_model = geospaNN.nngls_train(model, lr=0.01, min_delta=0.001)
training_log = nngls_model.train(data_train, data_val, data_test, Update_init=20, Update_step=10)
end_time = time.time()
print(f"NNGLS complete in {end_time - start_time:.2f}s")

# Predict & Plot 2: Truth vs Prediction
test_predict, test_U, test_L = model.predict(data_train, data_test, CI=True)
plt.figure(figsize=(6, 6))
plt.scatter(test_predict.detach().numpy(), data_test.y.detach().numpy(), s=1, label='Truth vs prediction', alpha=0.6)
plt.plot([data_test.y.min(), data_test.y.max()], [data_test.y.min(), data_test.y.max()], 'r--', lw=2, label='1:1 line')
plt.xlabel("Prediction")
plt.ylabel("Truth")
plt.legend()
plt.title('NNGLS Results')
plt.show()

# Plot 3: Maps
f_pred = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                         data_test.pos.detach().numpy()[:, 1])),
                                                test_predict.detach().numpy())
f_true = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                         data_test.pos.detach().numpy()[:, 1])),
                                                data_test.y.detach().numpy())
f_L = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                      data_test.pos.detach().numpy()[:, 1])),
                                             test_L.detach().numpy())
f_U = interpolate.CloughTocher2DInterpolator(list(zip(data_test.pos.detach().numpy()[:, 0],
                                                      data_test.pos.detach().numpy()[:, 1])),
                                             test_U.detach().numpy())

titles = [['Prediction', 'Truth'], ['Lower CI', 'Upper CI']]
fig, ax = plt.subplots(2, 2, figsize=(16, 12))
for i in range(2):
    for j in range(2):
        if i == 0 and j == 0:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_pred(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        elif i == 0 and j == 1:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_true(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
            ax[i,j].plot(data_test.pos.detach().numpy()[:,0], data_test.pos.detach().numpy()[:,1], 'o', c='orange', markersize=4)
        elif i == 1 and j == 0:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_L(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        else:
            im = ax[i,j].scatter(normalized_x_test, normalized_y_test, s=9, c=f_U(normalized_x_test, normalized_y_test),
                                 marker='s', alpha=0.7, vmin=0, vmax=20)
        ax[i,j].set_title(titles[i][j])
        plt.colorbar(im, ax=ax[i,j])
plt.tight_layout()
plt.show()

# PDP Plot
variable_names = ['Precipitation accumulation', 'Air temperature', 'Pressure', 'Relative humidity', 'U-wind', 'V-wind']
geospaNN.plot_PDP(model, X, variable_names)

print("Complete! All plots generated.")
data_PM25.head()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 6.3 MB/s eta 0:00:00
